## Milestone 1: Email Triage and Response Assistant


## 1.Load Environment Variables

In [ ]:
from dotenv import load_dotenv
import os

# Load API keys from .env
load_dotenv()

print("Gemini Key Loaded:", os.getenv("GOOGLE_API_KEY") is not None)
print("LangSmith Key Loaded:", os.getenv("LANGCHAIN_API_KEY") is not None)


Gemini Key Loaded: True
LangSmith Key Loaded: True


## 2. Initialize LLM

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0
)

# Test connection
response = llm.invoke("Say: Gemini connected successfully")
print(response.content)


Gemini connected successfully


## 3. Add Project Root to Python Path

In [3]:
import sys
import os

# Add project root to Python path
sys.path.append(os.path.abspath(".."))

print("Project root added to path")


Project root added to path


## 4. Import Tools

In [4]:
from src.tools import read_calendar, get_customer_profile

tools = {
    "read_calendar": read_calendar,
    "get_customer_profile": get_customer_profile
}


## 5. Define Triage Node

In [5]:
def triage_node(state):
    email = state["email"]
    prompt = f"""
Classify this email into one of:
ignore
notify_human
respond

Email:
{email}

Return only the label.
"""
    label = llm.invoke(prompt).content.strip().lower()
    return {**state, "triage": label}


## 6. Define React Agent

In [6]:
def react_agent(state):
    email = state["email"]
    prompt = f"""
You are an email assistant.
You can use tools if needed.

Tools:
read_calendar
get_customer_profile

Email:
{email}

If you need a tool, write TOOL:<toolname>

Otherwise give reply.
"""
    response = llm.invoke(prompt).content

    if "TOOL:" in response:
        tool_name = response.replace("TOOL:", "").strip()
        tool_result = tools[tool_name]()
        return {**state, "response": tool_result}

    return {**state, "response": response}


## 7. Build State Graph

In [7]:
from langgraph.graph import StateGraph

# Create a graph object
graph = StateGraph(dict)

# Add nodes
graph.add_node("triage", triage_node)
graph.add_node("react", react_agent)

# Conditional routing
def route(state):
    if state["triage"] == "respond":
        return "react"
    else:
        return "end"

graph.add_conditional_edges("triage", route)

# Set entry point
graph.set_entry_point("triage")

# Compile pipeline
app = graph.compile()


## 8. Load Sample Emails

In [8]:
import pandas as pd

emails = pd.read_csv("../data/sample_emails_with_triage_200.csv")  # use your correct CSV
emails.head()


,id,sender,subject,body,priority,triage_label,ideal_intent,ideal_tone
0,1,alerts@bank.com,Password Reset Request,Reminder: The client meeting is scheduled at 1...,low,notify_human,respond,neutral
1,2,alerts@bank.com,Congratulations! You've Won,Your invoice of INR 25515.09 is due on 2025-12...,low,respond,respond,urgent
2,3,no-reply@service.com,Promotion: Big Sale,Reminder: The client meeting is scheduled at 1...,low,ignore,respond,neutral
3,4,sales@shop.com,Monthly Report,"Hello team, please find the attached weekly re...",medium,respond,respond,neutral
4,5,no-reply@service.com,Survey,"Hello team, please find the attached weekly re...",low,respond,respond,neutral


## 9. Run Pipeline on First 25 Emails
First processing 20 emails with first key

In [9]:
results = []

for _, row in emails.head(25).iterrows():
    email = row["body"]
    output = app.invoke({"email": email})

    results.append({
        "email": email,
        "triage": output["triage"],
        "response": output.get("response", "")
    })

results[:3]


Task triage with path ('__pregel_pull', 'triage') wrote to unknown channel branch:to:end, ignoring it.
Task triage with path ('__pregel_pull', 'triage') wrote to unknown channel branch:to:end, ignoring it.
Task triage with path ('__pregel_pull', 'triage') wrote to unknown channel branch:to:end, ignoring it.
Task triage with path ('__pregel_pull', 'triage') wrote to unknown channel branch:to:end, ignoring it.
Task triage with path ('__pregel_pull', 'triage') wrote to unknown channel branch:to:end, ignoring it.
Task triage with path ('__pregel_pull', 'triage') wrote to unknown channel branch:to:end, ignoring it.
Task triage with path ('__pregel_pull', 'triage') wrote to unknown channel branch:to:end, ignoring it.
Task triage with path ('__pregel_pull', 'triage') wrote to unknown channel branch:to:end, ignoring it.
Task triage with path ('__pregel_pull', 'triage') wrote to unknown channel branch:to:end, ignoring it.
Task triage with path ('__pregel_pull', 'triage') wrote to unknown channe

ChatGoogleGenerativeAIError: Error calling model 'gemini-2.5-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 48.071854211s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeT

## 10. Save Milestone 1 Output
saved the output of first 20 emails

In [12]:

import pandas as pd

pd.DataFrame(results).to_csv("../data/milestone1_output.csv", index=False)


## 11. Resume from Saved Output
Loaded the saved milestone output and processed the remaining emails from 20- 25 by creating new key.

In [13]:
import pandas as pd

# Load your saved milestone output
final = pd.read_csv("../data/milestone1_output.csv")
results = final.to_dict("records")

print("Already processed:", len(results))


Already processed: 20


In [ ]:
# Resume processing next emails (index 20-24)
for _, row in emails.iloc[20:25].iterrows():
    email = row["body"]
    output = app.invoke({"email": email})

    results.append({
        "email": email,
        "triage": output["triage"],
        "response": output.get("response", "")
    })


Task triage with path ('__pregel_pull', 'triage') wrote to unknown channel branch:to:end, ignoring it.
Task triage with path ('__pregel_pull', 'triage') wrote to unknown channel branch:to:end, ignoring it.
Task triage with path ('__pregel_pull', 'triage') wrote to unknown channel branch:to:end, ignoring it.
Task triage with path ('__pregel_pull', 'triage') wrote to unknown channel branch:to:end, ignoring it.


## Save all results again

In [17]:
import pandas as pd

# Save everything into milestone1_output.csv
pd.DataFrame(results).to_csv("../data/milestone1_output.csv", index=False)


## 12. Verify Final Output
check the last 5 rows

In [ ]:
final_check = pd.read_csv("../data/milestone1_output.csv")
print("Total emails processed:", len(final_check))
final_check.tail(5)  # check the last 5 rows


Total emails processed: 25


,email,triage,response
20,Notice: Your account will be locked unless ver...,notify_human,NaN
21,Please complete the mandatory training module ...,notify_human,NaN
22,Your order #2545 has been shipped and is expec...,respond,Thank you for the update!
23,Security alert: multiple failed login attempts...,notify_human,NaN
24,Your invoice of INR 9983.71 is due on 2025-12-...,notify_human,NaN


## 13. Calculate Accuracy Against Golden Labels

In [22]:
import pandas as pd

# Load the expected labels
gold = pd.read_csv("../data/golden_labels.csv")

# Load your model output
pred = pd.read_csv("../data/milestone1_output.csv")

# Compare triage labels
accuracy = (gold["expected"] == pred["triage"]).mean()
print("Accuracy:", accuracy)


ValueError: Can only compare identically-labeled Series objects